# MFB 545 · Journée 3 · Scoring de crédit en direct

**Ce que vous allez faire, en 30 minutes :** charger un vrai jeu de données de crédit à la consommation (1 000 prêts, dont 300 en défaut), construire un score avec une régression logistique, mesurer sa qualité (AUC, Gini), puis choisir un **seuil d’acceptation** et voir ce qu’il fait au résultat du portefeuille.

**Mode d’emploi.** Exécutez chaque cellule de code dans l’ordre : cliquez dessus puis sur ▶ (ou `Maj + Entrée`). Aucune programmation n’est demandée : lisez les résultats et répondez aux questions **sur le site du cours** (section 3.2).

*Données : Hofmann, H. (1994), « Statlog (German Credit Data) », UCI Machine Learning Repository. Montants en Deutsche Mark de l’époque.*

## 1 · Charger les données

In [ ]:
import pandas as pd, numpy as np
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data"
colonnes = ["compte_courant","duree_mois","historique","objet","montant","epargne","emploi","taux_mensualite",
            "statut_sexe","garants","residence","patrimoine","age","autres_credits","logement","nb_credits",
            "metier","personnes_a_charge","telephone","travailleur_etranger","classe"]
df = pd.read_csv(url, sep=" ", header=None, names=colonnes)
df["defaut"] = (df["classe"] == 2).astype(int)        # 1 = le prêt a mal tourné
print(len(df), "prêts ·", df["defaut"].sum(), "défauts · taux de défaut", round(df["defaut"].mean()*100, 1), "%")
df.head()

## 2 · Une variable, à la main

Les codes de la colonne `compte_courant` : **A11** solde négatif · **A12** de 0 à 200 DM · **A13** 200 DM ou plus · **A14** pas de compte courant dans la banque. Regardez le taux de défaut par catégorie avant tout modèle : c’est ce qu’un analyste crédit faisait à la main avant les scores.

In [ ]:
df.groupby("compte_courant")["defaut"].agg(prets="count", taux_de_defaut="mean").round(3)

## 3 · Les variables du score

Sept informations, celles d’une demande de crédit ordinaire : l’état du compte courant (trois indicateurs 0/1, la référence étant « 200 DM ou plus »), la durée en mois, le montant en milliers, une épargne faible (moins de 100 DM), l’âge, et un historique de crédit défavorable.

**Le découpage.** On apprend sur les **700 premiers** prêts et on mesure sur les **300 derniers**, que le modèle n’a jamais vus. (D’habitude on tire ce découpage au hasard ; ici il est fixe pour que toute la classe — et le simulateur du site — obtienne exactement les mêmes chiffres.)

In [ ]:
X = pd.DataFrame({
    "compte_negatif":   (df.compte_courant == "A11").astype(int),
    "compte_0_200":     (df.compte_courant == "A12").astype(int),
    "pas_de_compte":    (df.compte_courant == "A14").astype(int),
    "duree_mois":       df.duree_mois,
    "montant_milliers": df.montant / 1000,
    "epargne_faible":   (df.epargne == "A61").astype(int),
    "age":              df.age,
    "historique_mauvais": df.historique.isin(["A30","A31"]).astype(int),
})
y = df.defaut
X_app, y_app, X_test, y_test = X.iloc[:700], y.iloc[:700], X.iloc[700:], y.iloc[700:]
X.head()

## 4 · La régression logistique

Le modèle calcule un **logit** = constante + Σ coefficient × variable, puis la probabilité de défaut **PD = 1 / (1 + e^(−logit))**. Un coefficient positif augmente le risque ; e^(coefficient) est le facteur par lequel la variable multiplie la *cote* de défaut (les « odds »).

In [ ]:
from sklearn.linear_model import LogisticRegression
modele = LogisticRegression(penalty=None, max_iter=5000).fit(X_app, y_app)
coef = pd.DataFrame({"coefficient": modele.coef_[0], "facteur_sur_la_cote": np.exp(modele.coef_[0])}, index=X.columns)
print("constante :", round(modele.intercept_[0], 4))
coef.round(4)

## 5 · Le score est-il bon ? AUC et Gini

L’**AUC** est la probabilité qu’un mauvais payeur tiré au hasard reçoive une PD plus élevée qu’un bon payeur tiré au hasard : 0,5 = pile ou face, 1 = parfait. Le **Gini** = 2 × AUC − 1 est la mesure que les banques publient. **Notez l’AUC sur l’échantillon de test : le site vous la demande.**

In [ ]:
from sklearn.metrics import roc_auc_score
pd_test = modele.predict_proba(X_test)[:, 1]
auc_test = roc_auc_score(y_test, pd_test)
auc_app  = roc_auc_score(y_app, modele.predict_proba(X_app)[:, 1])
print("AUC apprentissage :", round(auc_app, 3), "· AUC test :", round(auc_test, 3), "· Gini test :", round(2*auc_test - 1, 3))

## 6 · Le seuil contre la rentabilité

On accepte un demandeur si sa PD est **inférieure au seuil**. Hypothèses du simulateur du site : un prêt remboursé rapporte **15 %** de son montant (intérêts nets du coût de financement, sur la durée du prêt) ; un prêt en défaut fait perdre **50 %** de son montant (la perte en cas de défaut, LGD). Changez `marge` et `lgd` et réexécutez : le meilleur seuil bouge.

In [ ]:
marge, lgd = 0.15, 0.50
montants, defauts = df.montant.iloc[700:].values, y_test.values
lignes = []
for seuil in np.arange(0.05, 0.75, 0.05).round(2).tolist() + [1.01]:
    acc = pd_test < seuil
    gain = np.where(defauts[acc] == 0, marge * montants[acc], -lgd * montants[acc]).sum()
    lignes.append({"seuil": seuil, "acceptes": int(acc.sum()), "defauts": int(defauts[acc].sum()),
                   "taux_de_defaut_%": round(100 * defauts[acc].mean(), 1) if acc.any() else 0, "resultat_DM": round(gain)})
table = pd.DataFrame(lignes); table

In [ ]:
import matplotlib.pyplot as plt
seuils = np.arange(0.01, 0.80, 0.01)
res = [np.where(defauts[pd_test < s] == 0, marge*montants[pd_test < s], -lgd*montants[pd_test < s]).sum() for s in seuils]
plt.figure(figsize=(8, 4)); plt.plot(seuils*100, res); plt.axhline(0, color="grey", lw=.8)
plt.axvline(100*marge/(marge+lgd), ls="--", color="red", label="seuil théorique = marge / (marge + LGD)")
plt.xlabel("Seuil d’acceptation : PD maximale (%)"); plt.ylabel("Résultat du portefeuille (DM)"); plt.legend(); plt.show()
print("Meilleur seuil :", round(seuils[int(np.argmax(res))]*100), "% · résultat", round(max(res)), "DM")

## 7 · Ce que le jeu de données contient aussi

Il contient le **sexe et l’état civil** (`statut_sexe` : A91 homme divorcé/séparé · A92 femme divorcée/séparée/mariée · A93 homme célibataire · A94 homme marié/veuf) et la **nationalité étrangère** (`travailleur_etranger` : A201 oui · A202 non). Le modèle ci-dessus ne les utilise pas. Regardez les taux de défaut par groupe, puis répondez sur le site : *les utiliseriez-vous si elles amélioraient l’AUC ?* Dans l’UE et en Suisse, la réponse est non (non-discrimination, protection des données ; et l’AI Act classe le scoring de crédit des personnes comme un système d’IA **à haut risque**). Mais une variable autorisée peut en reproduire une interdite : c’est le problème des **variables proxy**.

In [ ]:
for col in ["statut_sexe", "travailleur_etranger"]:
    print(df.groupby(col)["defaut"].agg(prets="count", taux_de_defaut="mean").round(3), "\n")

## À reporter sur le site (section 3.2)

1. L’**AUC sur le test** (cellule 5).
2. La variable dont le **facteur sur la cote** est le plus élevé, et ce qu’il veut dire en une phrase (cellule 4).
3. Le **meilleur seuil** et le résultat correspondant (cellule 6) — puis refaites-le dans le simulateur de la page, avec vos propres hypothèses.